# Basic analysis of mtg card data

Using the json provided by scryfall. starting @ 3/4/2026

In [1]:
import seaborn as sns
import pandas as pd
# import numpy as np


First we load the json of all cards with unique oracle identification numbers. So we are only getting one object per unique card, not unique printings.

In [2]:
cards = pd.read_json('../oracle-cards-20260305100222.json')

Now we can get rid of the unnecessary columns. Think, the artist and whether or not it is a showcase or a foil.

In [ ]:
# detail the columns that we are going to drop from the list because they
to_drop = ['id', 'multiverse_ids', 'mtgo_id',
       'tcgplayer_id', 'cardmarket_id', 'lang', 'released_at', 'uri',
       'scryfall_uri', 'layout', 'highres_image', 'image_status', 'image_uris',
       'games', 'reserved', 'foil', 'nonfoil', 'finishes',
       'oversized', 'promo', 'reprint', 'variation', 'set_uri', 'set_search_uri', 'scryfall_set_uri',
       'rulings_uri', 'prints_search_uri', 'collector_number', 'flavor_text', 'artist',
       'artist_ids', 'illustration_id', 'border_color', 'frame',
       'frame_effects', 'security_stamp', 'full_art', 'textless', 'booster',
       'story_spotlight', 'preview', 'related_uris',
       'purchase_uris', 'mtgo_foil_id', 'penny_rank', 'arena_id',
       'promo_types', 'tcgplayer_etched_id', 'attraction_lights', 'color_indicator', 'flavor_name',
       'content_warning', 'printed_type_line', 'defense', 'printed_text']

# drop the columns detailed above
simplified_cards = cards.drop(to_drop, axis=1)

# set the index to be the oracle_id for consistency
simplified_cards.set_index("oracle_id", inplace=True)

Now we drop each column where the card does not have an edhrecrank and convert some of the columns to simpler 

In [8]:
simplified_cards = simplified_cards[simplified_cards["edhrec_rank"] > 0]

simplified_cards["name"] = simplified_cards["name"].astype(str)

simplified_cards.info()

<class 'pandas.core.frame.DataFrame'>
Index: 30722 entries, 00037840-6089-42ec-8c5c-281f9f474504 to ffff90c3-63c4-4dee-a21d-6b2b113f4f80
Data columns (total 30 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   object          30722 non-null  object 
 1   name            30722 non-null  object 
 2   mana_cost       30240 non-null  object 
 3   cmc             30722 non-null  float64
 4   type_line       30722 non-null  object 
 5   oracle_text     29963 non-null  object 
 6   power           16905 non-null  object 
 7   toughness       16905 non-null  object 
 8   colors          30240 non-null  object 
 9   color_identity  30722 non-null  object 
 10  keywords        30722 non-null  object 
 11  all_parts       5117 non-null   object 
 12  legalities      30722 non-null  object 
 13  game_changer    30722 non-null  bool   
 14  set_id          30722 non-null  object 
 15  set             30722 non-null  object 
 16  set_name       

Now we get the datatypes for the color identity column

In [5]:
simplified_cards.dtypes

object             object
name               object
mana_cost          object
cmc               float64
type_line          object
oracle_text        object
power              object
toughness          object
colors             object
color_identity     object
keywords           object
all_parts          object
legalities         object
game_changer         bool
set_id             object
set                object
set_name           object
set_type           object
digital              bool
rarity             object
watermark          object
card_back_id       object
edhrec_rank       float64
prices             object
card_faces         object
produced_mana      object
resource_id        object
loyalty            object
life_modifier     float64
hand_modifier     float64
dtype: object

Now we will examine the colors of each card and establish a new column that is
indicative of how many colors a column has.

then we can sort for the actual colors based on a lambda expression that helps to "search" 
the list of colors that the color identity column actually looks like.

In [ ]:
# To see how many colors each card has
simplified_cards['color_count'] = simplified_cards['color_identity'].str.len()

# To check if a card is "Red" or "Green" inclusive
is_white = simplified_cards['color_identity'].apply(lambda x: 'W' in x)
is_blue = simplified_cards['color_identity'].apply(lambda x: 'U' in x)
is_black = simplified_cards['color_identity'].apply(lambda x: 'B' in x)
is_red = simplified_cards['color_identity'].apply(lambda x: 'R' in x)
is_green = simplified_cards['color_identity'].apply(lambda x: 'G' in x)

cards_by_color = {'White': simplified_cards[is_white], 
                  'Blue': simplified_cards[is_blue], 
                  'Black': simplified_cards[is_black], 
                  'Red': simplified_cards[is_red], 
                  'Green': simplified_cards[is_green]}


White
                                     object                     name  \
oracle_id                                                              
0012bc78-e69d-4a67-a302-e5fe0dfd4407   card           Ravnica at War   
002a9cea-8cf7-48ba-83eb-e1c87a7024e5   card           Disposal Mummy   
003c48d4-0303-4970-8637-47ae060385aa   card         Safewright Quest   
00573e77-8ff6-4acb-8683-8827d965288f   card          Behemoth Sledge   
005c181a-05db-4c15-9893-65103cca338e   card  Toluz, Clever Conductor   

                                          mana_cost  cmc  \
oracle_id                                                  
0012bc78-e69d-4a67-a302-e5fe0dfd4407         {3}{W}  4.0   
002a9cea-8cf7-48ba-83eb-e1c87a7024e5         {2}{W}  3.0   
003c48d4-0303-4970-8637-47ae060385aa          {G/W}  1.0   
00573e77-8ff6-4acb-8683-8827d965288f      {1}{G}{W}  3.0   
005c181a-05db-4c15-9893-65103cca338e  {W/U}{U}{U/B}  3.0   

                                                             type_li